Two Crude Problem Section
 2.1 / 2.2

In [1]:
using LinearAlgebra

In [2]:
@enum OptimizationGoal MINIMIZE MAXIMIZE
@enum TerminationStatus PENDING OPTIMAL UNBOUNDED INFEASIBLE

Base.@kwdef mutable struct Model
    goal::OptimizationGoal = MINIMIZE
    c::Vector{Float64} = Float64[]
    A::Matrix{Float64} = Matrix{Float64}(undef, 0, 0)
    b::Vector{Float64} = Float64[]
    basic_vars::Vector{Int} = Int[]
    non_basic_vars::Vector{Int} = Int[]
    termination_status::TerminationStatus = PENDING
    solution::Vector{Float64} = Float64[]
    objective_value::Float64 = 0.0
end

Model

In [3]:
model = Model()

model.goal = MINIMIZE

model.c = [100.0, 75.0, 0.0, 0.0, 0., 0.0, 0.0]

model.A = [0.3 0.4 -1.0  0.0  0.0 0.0 0.0;
           0.4 0.2  0.0 -1.0  0.0 0.0 0.0;
           0.2 0.3  0.0  0.0 -1.0 0.0 0.0;
           1.0 0.0  0.0  0.0  0.0 1.0 0.0;
           0.0 1.0  0.0  0.0  0.0 0.0 1.0]
        
model.b = [2.0, 1.5, 0.5, 9.0, 6.0]

model.basic_vars = collect(1:5)
model.non_basic_vars = collect(6:7)

model

Model(MINIMIZE, [100.0, 75.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.3 0.4 … 0.0 0.0; 0.4 0.2 … 0.0 0.0; … ; 1.0 0.0 … 1.0 0.0; 0.0 1.0 … 0.0 1.0], [2.0, 1.5, 0.5, 9.0, 6.0], [1, 2, 3, 4, 5], [6, 7], PENDING, Float64[], 0.0)

In [10]:
function simplex(model::Model; verbose=true)
    A_b = model.A[:, model.basic_vars]
    x_b = A_b \ model.b

    model.solution = zeros(length(model.c))
    model.solution[model.basic_vars] = x_b

    if verbose
        println("Initial solution x: ", model.solution)
        println("Initial basic variables: ", model.basic_vars)
        println("Initial non-basic variables: ", model.non_basic_vars)
        println("Initial objective value c * x: ", model.c ⋅ model.solution)
    end

    solving = true
    iteration = 0
    termination_status = PENDING
    while solving
        # lnbv = leaving nonbasic variable
        lnbv = 0
        lnbv_idx = 0
        optimum_improvement = model.goal == MINIMIZE ? 1 : -1
        optimum_path = []

        for (nbv_idx, nbv) ∈ enumerate(model.non_basic_vars)
            path = zeros(length(model.c))
            path[nbv] = 1
            path[model.basic_vars] = model.A[:, model.basic_vars] \ (-model.A[:, nbv])
            println("Path: ", path)
            path_improvement = model.c ⋅ path
            println("Improvement: ", path_improvement)
            if (model.goal == MAXIMIZE && path_improvement > optimum_improvement) ||
               (model.goal == MINIMIZE && path_improvement < optimum_improvement)
                optimum_improvement = path_improvement
                lnbv = nbv
                lnbv_idx = nbv_idx
                optimum_path = path
            end
        end

        if lnbv == 0
            solving = false
            model.termination_status = OPTIMAL
            model.objective_value = round(model.c ⋅ model.solution, digits=2)
            break   
        end

        iteration += 1

        if verbose
            println("Iteration: ", iteration)

            println("Leaving nonbasic variable: ", lnbv)
            println("Optimum path: ", optimum_path)
            println("Optimum improvement: ", optimum_improvement)
        end

        # Determine largest step along the best path
        ratios = Dict{Int, Float64}()
        for (bv_idx, bv) ∈ enumerate(model.basic_vars)
            if optimum_path[bv] < 0
                ratios[bv_idx] = -model.solution[bv] / optimum_path[bv]
            end
        end

        if isempty(ratios)
            solving = false
            termination_status = UNBOUNDED
            break
        end
        step, lbv_idx = findmin(ratios)
        lbv = model.basic_vars[lbv_idx]
        if verbose
            println("Ratios: ", ratios)
            println("Step: ", step)
            println("Leaving basic variable: ", lbv)
        end

        # Determine the new solution after taking the step along the best path
        x1 = model.solution + step * optimum_path
        println("New solution: ", x1)

        # Update the basic and non-basic variables after the pivot
        model.basic_vars[lbv_idx] = lnbv
        sort!(model.basic_vars)
        model.non_basic_vars[lnbv_idx] = lbv
        sort!(model.non_basic_vars)
        model.solution = x1
        if verbose
            println("Updated solution x: ", model.solution)
            println("Updated basic variables: ", model.basic_vars)
            println("Updated non-basic variables: ", model.non_basic_vars)
            println("Updated objective value c * x: ", model.c ⋅ model.solution)
            println()
        end
    end
end

simplex(model; verbose=false)
println("Final model: ", model)

Path: [-2.0, 4.0, 1.0, 0.0, 0.7999999999999999, 2.0, -4.0]
Improvement: 100.0
Path: [4.0, -3.0, 0.0, 1.0, -0.09999999999999987, -4.0, 3.0]
Improvement: 175.0
Final model: Model(MINIMIZE, [100.0, 75.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.3 0.4 -1.0 0.0 0.0 0.0 0.0; 0.4 0.2 0.0 -1.0 0.0 0.0 0.0; 0.2 0.3 0.0 0.0 -1.0 0.0 0.0; 1.0 0.0 0.0 0.0 0.0 1.0 0.0; 0.0 1.0 0.0 0.0 0.0 0.0 1.0], [2.0, 1.5, 0.5, 9.0, 6.0], [1, 2, 5, 6, 7], [3, 4], OPTIMAL, [2.0, 3.5000000000000004, 0.0, 0.0, 0.95, 7.0, 2.4999999999999996], 462.5)


In [9]:
function Base.show(io::IO, ::MIME"text/plain", m::Model)
    printstyled(io, "Model:\n", color=:blue, bold=true)
    for field in fieldnames(Model)
        val = getfield(m, field)
        
        print(io, "  ")
        printstyled(io, lpad(string(field), 18), color=:yellow, bold=true)
        println(io, " : ", val)
    end
end

display(model)

Model:
                goal : MINIMIZE
                   c : [100.0, 75.0, 0.0, 0.0, 0.0, 0.0, 0.0]
                   A : [0.3 0.4 … 0.0 0.0; 0.4 0.2 … 0.0 0.0; … ; 1.0 0.0 … 1.0 0.0; 0.0 1.0 … 0.0 1.0]
                   b : [2.0, 1.5, 0.5, 9.0, 6.0]
          basic_vars : [1, 2, 5, 6, 7]
      non_basic_vars : [3, 4]
  termination_status : OPTIMAL
            solution : [2.0, 3.5000000000000004, 0.0, 0.0, 0.95, 7.0, 2.4999999999999996]
     objective_value : 462.5
